# Mendelian Randomization with MRBIGR2

Two-sample MR workflow: format instruments, estimate causal effects, sensitivity analyses, and functional enrichment.

**Prerequisites:** `pip install -e .` from the MRBIGR2 repo root.

In [ ]:
import os
import pandas as pd
from mrbigr.core import mr, go, qtl

OUT_DIR = "output/mr_demo"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Prepare instrument variables

Start from QTL results or pre-formatted GWAS summary statistics.

In [ ]:
# Option A: format from QTL file
QTL_FILE = "data/example_qtl.csv"  # adjust to your data
if os.path.exists(QTL_FILE):
    instruments = mr.format_qtl_for_mr(QTL_FILE, output_file=f"{OUT_DIR}/instruments.csv")
    display(instruments.head())
else:
    print(f"QTL file not found at {QTL_FILE}, using example data below.")

In [ ]:
# Option B: load pre-formatted summary stats
exposure = pd.DataFrame({
    "rs": ["rs1", "rs2", "rs3", "rs4", "rs5"],
    "beta": [0.15, -0.08, 0.22, 0.10, -0.12],
    "se":   [0.03,  0.02, 0.04, 0.03,  0.03],
})
outcome = pd.DataFrame({
    "rs": ["rs1", "rs2", "rs3", "rs4", "rs5"],
    "beta": [0.05, -0.03, 0.08, 0.04, -0.05],
    "se":   [0.02,  0.01, 0.02, 0.02,  0.02],
})

## 2. IVW causal estimate

In [ ]:
result = mr.mr_analysis(exposure, outcome, method="ivw")
display(result)

In [ ]:
causal = mr.mr_causal_estimate(
    exposure, outcome,
    snp_col="rs", beta_col="beta", se_col="se", method="ivw"
)
display(causal)

## 3. Sensitivity analyses

In [ ]:
# Pleiotropy test (MR-Egger intercept)
pleiotropy = mr.test_pleiotropy(exposure, outcome)
print("Pleiotropy test:", pleiotropy)

# Heterogeneity test (Cochran's Q)
heterogeneity = mr.heterogeneity_test(exposure, outcome)
print("Heterogeneity test:", heterogeneity)

## 4. Functional enrichment (optional)

Run GO and KEGG enrichment on causal candidate genes.

In [ ]:
causal_genes = ["Gene1", "Gene2", "Gene3"]  # replace with real gene list

go_result = go.go_enrich(
    pd.DataFrame({"genes": causal_genes}),
    organism="Mouse", mode="local"
)
if go_result is not None:
    display(go_result.head(10))
else:
    print("No significant GO terms found.")